In [10]:
import pandas as pd

In [11]:
url_periodos = (
    "https://www.datos.gov.co/resource/kgxf-xxbe.csv"
    "?$select=periodo"
    "&$group=periodo"
    "&$order=periodo DESC"
    "&$limit=20"
)
url_periodos = url_periodos.replace(" ", "%20")

periodos_recientes = pd.read_csv(url_periodos)
periodos_recientes

,periodo
0,20224
1,20221
2,20211
3,20201
4,20194
5,20191
6,20181
7,20172
8,20171
9,20162


In [12]:
lista_periodos = periodos_recientes['periodo'].tolist()

periodos_ultimos_2_anos = lista_periodos[:4]

print("Periodos que se van a usar:", periodos_ultimos_2_anos)

Periodos que se van a usar: [20224, 20221, 20211, 20201]


In [13]:
prueba_zona = pd.read_csv("https://www.datos.gov.co/resource/kgxf-xxbe.csv?$limit=1")
prueba_zona.columns.tolist()

['periodo',
 'estu_tipodocumento',
 'estu_consecutivo',
 'cole_area_ubicacion',
 'cole_bilingue',
 'cole_calendario',
 'cole_caracter',
 'cole_cod_dane_establecimiento',
 'cole_cod_dane_sede',
 'cole_cod_depto_ubicacion',
 'cole_cod_mcpio_ubicacion',
 'cole_codigo_icfes',
 'cole_depto_ubicacion',
 'cole_genero',
 'cole_jornada',
 'cole_mcpio_ubicacion',
 'cole_naturaleza',
 'cole_nombre_establecimiento',
 'cole_nombre_sede',
 'cole_sede_principal',
 'estu_cod_depto_presentacion',
 'estu_cod_mcpio_presentacion',
 'estu_cod_reside_depto',
 'estu_cod_reside_mcpio',
 'estu_depto_presentacion',
 'estu_depto_reside',
 'estu_estadoinvestigacion',
 'estu_estudiante',
 'estu_fechanacimiento',
 'estu_genero',
 'estu_mcpio_presentacion',
 'estu_mcpio_reside',
 'estu_nacionalidad',
 'estu_pais_reside',
 'estu_privado_libertad',
 'fami_cuartoshogar',
 'fami_educacionmadre',
 'fami_educacionpadre',
 'fami_estratovivienda',
 'fami_personashogar',
 'fami_tieneautomovil',
 'fami_tienecomputador',
 'fam

In [14]:
periodos_texto = ",".join([f"'{p}'" for p in periodos_ultimos_2_anos])

url_saber11 = (
    "https://www.datos.gov.co/resource/kgxf-xxbe.csv"
    "?$select=cole_mcpio_ubicacion,cole_area_ubicacion,punt_matematicas,punt_c_naturales"
    f"&$where=periodo in({periodos_texto})"
    "&$limit=500000"
)
url_saber11 = url_saber11.replace(" ", "%20")

saber11_crudo = pd.read_csv(url_saber11)

print(f"Filas descargadas: {saber11_crudo.shape[0]}")
saber11_crudo.head()

Filas descargadas: 500000


,cole_mcpio_ubicacion,cole_area_ubicacion,punt_matematicas,punt_c_naturales
0,CALI,URBANO,76,71
1,BOGOTÁ D.C.,URBANO,73,75
2,CALI,URBANO,62,55
3,LA ESTRELLA,RURAL,74,61
4,CALI,URBANO,32,48


In [15]:
saber11 = (
    saber11_crudo
    .rename(columns={
        "cole_mcpio_ubicacion": "municipio",
        "cole_area_ubicacion": "zona"
    })
    .groupby(["municipio", "zona"], as_index=False)
    .agg(
        promedio_ciencias=("punt_c_naturales", "mean"),
        promedio_matematicas=("punt_matematicas", "mean")
    )
)

print(f"Filas (municipio + zona): {saber11.shape[0]}")
saber11.head()

Filas (municipio + zona): 1888


,municipio,zona,promedio_ciencias,promedio_matematicas
0,ABEJORRAL,RURAL,46.906667,47.920000
1,ABEJORRAL,URBANO,53.507246,57.318841
2,ABRIAQUÍ,URBANO,44.625000,47.375000
3,ACACÍAS,RURAL,51.575419,53.754190
4,ACACÍAS,URBANO,51.103448,53.640180


In [19]:
url_conectividad = "https://www.postdata.gov.co/resource/797/download/file"

conectividad_crudo = pd.read_csv(
    url_conectividad,
    usecols=["MUNICIPIO", "ACCESOS"],
    sep=';'
)

print(f"Filas descargadas: {conectividad_crudo.shape[0]}")
conectividad_crudo.head()

Filas descargadas: 973335


,MUNICIPIO,ACCESOS
0,MORALES,3
1,MORALES,1
2,MORALES,1
3,PIENDAMÓ,3
4,PIENDAMÓ,3


In [21]:
conectividad_crudo['MUNICIPIO'] = conectividad_crudo['MUNICIPIO'].replace({
    'BOGOTÁ': 'BOGOTA',
    'BOGOTÁ, D.C.': 'BOGOTA'
})

conectividad = (
    conectividad_crudo
    .groupby("MUNICIPIO", as_index=False)
    .agg(total_accesos_internet=("ACCESOS", "sum"))
    .rename(columns={"MUNICIPIO": "municipio"})
)

print(f"Municipios: {conectividad.shape[0]}")
print("Bogotá en conectividad:", [m for m in conectividad['municipio'].unique() if 'BOGOT' in m])
conectividad.head()

Municipios: 1078


,municipio,total_accesos_internet
0,ABEJORRAL,5812
1,ABREGO,21912
2,ABRIAQUÍ,1825
3,ACACÍAS,215070
4,ACANDÍ,14099


In [22]:
saber11 = saber11.dropna().drop_duplicates()
conectividad = conectividad.dropna().drop_duplicates()

print(f"Saber 11: {saber11.shape[0]} filas")
print(f"Conectividad: {conectividad.shape[0]} filas")

Saber 11: 1888 filas
Conectividad: 1078 filas


In [23]:
saber11['municipio'] = saber11['municipio'].str.strip().str.upper()
conectividad['municipio'] = conectividad['municipio'].str.strip().str.upper()

print("Ejemplos Saber 11:", saber11['municipio'].unique()[:5])
print("Ejemplos conectividad:", conectividad['municipio'].unique()[:5])

Ejemplos Saber 11: ['ABEJORRAL' 'ABRIAQUÍ' 'ACACÍAS' 'ACANDÍ' 'ACEVEDO']
Ejemplos conectividad: ['ABEJORRAL' 'ABREGO' 'ABRIAQUÍ' 'ACACÍAS' 'ACANDÍ']


In [24]:
saber11['municipio'] = saber11['municipio'].replace({'BOGOTÁ D.C.': 'BOGOTA'})

print("Bogotá en Saber 11:", [m for m in saber11['municipio'].unique() if 'BOGOT' in m])
print("Bogotá en conectividad:", [m for m in conectividad['municipio'].unique() if 'BOGOT' in m])

Si no tuviste diferencias, puedes ejecutar esta celda vacía y seguir sin problema.


In [ ]:
final = pd.merge(saber11, conectividad, on='municipio', how='left')

print(f"Tabla final: {final.shape[0]} filas — {final.shape[1]} columnas")
final.head(10)

In [ ]:
final.to_csv('stemdata_limpio.csv', index=False, encoding='utf-8-sig')

from google.colab import files
files.download('stemdata_limpio.csv')

print("Archivo generado y descargado")